In [1]:
import uproot
import numpy as np
import pandas as pd
import awkward as ak
import matplotlib.pyplot as plt
import pickle
import os
from scipy.constants import c
import sys

sys.path.append("..")

DATA_DIR = "../mu3e_trigger_data"
ROOT_DATA_DIR = "../mu3e_root_data"
signal_data_file = f"{ROOT_DATA_DIR}/run_sig_simi-sort.root"
background_data_file = f"{ROOT_DATA_DIR}/run_bg_simi-sort.root"
e5_data_file = f"{ROOT_DATA_DIR}/run42_5e-sort.root"
familong_data_file = f"{ROOT_DATA_DIR}/run42_familon-sort.root"
signal_only_data_file = f"{ROOT_DATA_DIR}/run42_sig_only-sort.root"

HIT_COUNT_CUTOFF = 256

In [ ]:
from keras_src.data_preparation import convert_root_to_npy, get_image_slices_from_root

In [3]:
if True:
    convert_root_to_npy(
        file_path=signal_data_file,
        out_dir=DATA_DIR,
        out_name="sig",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
        add_layer_as_feature=True,
    )
    convert_root_to_npy(
        file_path=background_data_file,
        out_dir=DATA_DIR,
        out_name="bg",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
        add_layer_as_feature=True,
    )
    convert_root_to_npy(
        file_path=e5_data_file,
        out_dir=DATA_DIR,
        out_name="5e",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
        add_layer_as_feature=True,
    )
    convert_root_to_npy(
        file_path=familong_data_file,
        out_dir=DATA_DIR,
        out_name="familon",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
        add_layer_as_feature=True,
    )
    convert_root_to_npy(
        file_path=signal_only_data_file,
        out_dir=DATA_DIR,
        out_name="sig_only",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
        add_layer_as_feature=True,
    )

KeyboardInterrupt: 

In [ ]:
if False:
    get_image_slices_from_root(
        file_path=signal_data_file,
        out_dir=DATA_DIR,
        out_name="sig",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
    )
    get_image_slices_from_root(
        file_path=background_data_file,
        out_dir=DATA_DIR,
        out_name="bg",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
    )
    get_image_slices_from_root(
        file_path=e5_data_file,
        out_dir=DATA_DIR,
        out_name="e5",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
    )
    get_image_slices_from_root(
        file_path=familong_data_file,
        out_dir=DATA_DIR,
        out_name="familon",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
    )
    get_image_slices_from_root(
        file_path=signal_only_data_file,
        out_dir=DATA_DIR,
        out_name="sig_only",
        padding_value=-1,
        hit_cutoff=HIT_COUNT_CUTOFF,
    )

In [ ]:
def convert_pixel_id_to_nla(
    pixel_id: np.ndarray, padding_value: int = -1
) -> np.ndarray:
    nla = np.full((*pixel_id.shape, 4), padding_value, dtype=np.int32)
    valid_mask = pixel_id != padding_value

    chip_id = pixel_id // 2**16
    station = chip_id // 2**12
    layer = ((chip_id // 2**10) % 4) + 1
    phi = ((chip_id // 2**5) % 2**5) + 1
    z_prime = chip_id % 2**5

    z = np.where(layer == 3, z_prime - 7, np.where(layer == 4, z_prime - 6, z_prime))

    station_mask = station == 0
    valid_mask = valid_mask & station_mask

    nla[valid_mask, 0] = station[valid_mask]
    nla[valid_mask, 1] = layer[valid_mask]
    nla[valid_mask, 2] = phi[valid_mask]
    nla[valid_mask, 3] = z[valid_mask]

    return nla